# Breast Cancer Detection — Notebook 4
## Vision Transformer (ViT-Base/16) on BreakHis Histopathology

Fine-tunes an ImageNet-pretrained **ViT-Base/16** (via `timm`) on the **BreakHis** breast tumour histopathology dataset (100× magnification). The split is **patient-wise** (slide IDs parsed from the filenames) to avoid leakage between train and test.

### 1. Setup

In [ ]:
!pip install -q timm

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive/breast_cancer_project"

for folder in ["data", "models", "results", "plots"]:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Project folders ready")

In [ ]:
from google.colab import files
files.upload()   # upload your kaggle.json here

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### 2. Dataset — BreakHis (100× magnification)

In [ ]:
!kaggle datasets download -d ambarish/breakhis
!unzip -q breakhis.zip -d data

In [ ]:
import os
from glob import glob

all_paths = glob("data/**/*.png", recursive=True)
print("All magnifications:", len(all_paths))

# keep the 100X magnification only (common choice in the BreakHis literature)
paths = [p for p in all_paths if "/100X/" in p]
print("100X images:", len(paths))
print("Example:", paths[0])

In [ ]:
import pandas as pd

def get_patient_id(path):
    # filename e.g.  SOB_B_TA-14-4659-100-001.png  ->  patient/slide id 14-4659
    name = os.path.basename(path)
    parts = name.split("-")
    return parts[1] + "-" + parts[2]

df = pd.DataFrame({"path": paths})
df["label"] = df["path"].apply(lambda p: 1 if "/malignant/" in p else 0)
df["patient"] = df["path"].apply(get_patient_id)

print(df["label"].value_counts())
print("Patients:", df["patient"].nunique())
df.head()

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
from sklearn.model_selection import train_test_split

# -------- patient-wise split, stratified by the patient's diagnosis --------
patient_df = df.groupby("patient")["label"].first().reset_index()

train_pat, temp_pat = train_test_split(patient_df, test_size=0.30,
                                       stratify=patient_df["label"], random_state=42)
val_pat, test_pat = train_test_split(temp_pat, test_size=0.50,
                                     stratify=temp_pat["label"], random_state=42)

train_df = df[df["patient"].isin(train_pat["patient"])]
val_df = df[df["patient"].isin(val_pat["patient"])]
test_df = df[df["patient"].isin(test_pat["patient"])]

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))
print("Malignant ratio (train):", round(train_df["label"].mean(), 3))

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class BreakHisDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

train_loader = DataLoader(BreakHisDataset(train_df, train_transform),
                          batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(BreakHisDataset(val_df, val_transform),
                        batch_size=16, num_workers=2)
test_loader = DataLoader(BreakHisDataset(test_df, val_transform),
                         batch_size=16, num_workers=2)

### 3. Model — ViT-Base/16 (`timm`)

In [ ]:
import timm
import torch.nn as nn

model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# class weighting for the benign/malignant imbalance
n_pos = int(train_df["label"].sum())
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], device=device)
print("pos_weight:", pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# transformers prefer AdamW + low LR + cosine decay
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

### 4. Training

In [ ]:
EPOCHS = 8
history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().to(device)

        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item():.4f}")
    scheduler.step()

    # ---------------- validation ----------------
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()

            preds = (torch.sigmoid(outputs) > 0.5).int()
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), base_path + "/models/vit_breakhis_best.pth")
        print("  -> new best model saved")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], color="#0072B2", lw=2, label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], color="#D55E00", lw=2, label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()

axes[1].plot(epochs_range, history["val_acc"], color="#009E73", lw=2, label="Val accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/vit_breakhis_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Evaluation on held-out patients

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, accuracy_score)

model.load_state_dict(torch.load(base_path + "/models/vit_breakhis_best.pth"))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(np.array(labels))

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
preds = (all_probs > 0.5).astype(int)

acc = accuracy_score(all_labels, preds)
auc = roc_auc_score(all_labels, all_probs)

print("Test Accuracy:", round(acc, 4))
print("ROC-AUC:", round(auc, 4))
print("\nConfusion Matrix:\n", confusion_matrix(all_labels, preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, preds,
                            target_names=["benign", "malignant"]))

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion matrix (single-hue sequential colormap)
cm = confusion_matrix(all_labels, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["benign", "malignant"],
            yticklabels=["benign", "malignant"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="#0072B2", lw=2, label=f"ROC (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Chance")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/vit_breakhis_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Save artifacts to Drive

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
f1 = f1_score(all_labels, preds)

results_text = f"""MODEL: Vision Transformer (ViT-Base/16)
DATASET: BreakHis 100X (patient-wise split)
LOSS: BCEWithLogitsLoss (pos_weight from data)

Test Accuracy: {acc:.4f}
ROC-AUC: {auc:.4f}

Malignant class:
Precision: {precision:.4f}
Recall: {recall:.4f}
F1-score: {f1:.4f}

Confusion Matrix:
{confusion_matrix(all_labels, preds)}

Classification Report:
{classification_report(all_labels, preds)}
"""

with open(base_path + "/results/vit_breakhis_results.txt", "w") as f:
    f.write(results_text)

print(results_text)
print("Results saved")

In [ ]:
results_df = pd.DataFrame({
    "path": test_df["path"].values,
    "true_label": all_labels,
    "predicted_label": preds,
    "probability": all_probs
})

results_df.to_csv(base_path + "/results/vit_breakhis_predictions.csv", index=False)
print("Predictions saved")

In [ ]:
import json

config = {
    "model": "vit_base_patch16_224 (timm)",
    "pretrained": "ImageNet",
    "dataset": "BreakHis (ambarish/breakhis), 100X magnification",
    "task": "binary classification (malignant vs benign)",
    "split": "patient-wise 70/15/15, stratified",
    "loss": "BCEWithLogitsLoss + pos_weight",
    "optimizer": "AdamW",
    "lr": 2e-5,
    "weight_decay": 0.01,
    "scheduler": "CosineAnnealingLR",
    "epochs": 8,
    "batch_size": 16,
    "image_size": 224,
    "threshold": 0.5,
    "seed": 42
}

with open(base_path + "/results/vit_breakhis_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Config saved")